----------------------
#### Callbacks
---------------------

**What is a callback in LangChain?**

A callback in LangChain is a mechanism that allows you to monitor, log, or react to events during the execution of chains, agents, or models. 

Callbacks can capture information such as when a prompt is sent, when a response is received, errors, token usage, and more. 

They are useful for debugging, tracking, and customizing the behavior of your LLM-powered workflows.

### Examples of Callbacks in LangChain

Below are five practical examples of using callbacks in LangChain, each with code snippets.

In [1]:
# Simple Print Callback: Log when LLM starts and ends
from langchain.callbacks.base import BaseCallbackHandler

In [10]:
class PrintCallbackHandler(BaseCallbackHandler):
    
    def on_llm_start(self, *args, **kwargs):
        print("LLM started!")
        
    def on_llm_end(self, *args, **kwargs):
        print("LLM finished!")

In [11]:
handler = PrintCallbackHandler()

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo", callbacks=[handler])

llm.invoke("Say hello!")

LLM started!
LLM finished!
LLM finished!


AIMessage(content='Hello! How can I help you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 10, 'total_tokens': 19, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-CgtigZDcWVNgcj3zTy7K0IArSbvdw', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--5e3f2a34-9360-4eb9-ada5-13e2b52f8969-0', usage_metadata={'input_tokens': 10, 'output_tokens': 9, 'total_tokens': 19, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [12]:
# Token Usage Callback: Track token usage for each call
from langchain.callbacks.base import BaseCallbackHandler

In [18]:
class TokenUsageCallbackHandler(BaseCallbackHandler):
    def on_llm_end(self, response, **kwargs):
        # Try to get token usage from llm_output
        usage = getattr(response, 'llm_output', {}).get('token_usage', None)
        if usage:
            print("Token usage:", usage)
        else:
            # Try to get from generations/messages if available
            try:
                gen = response.generations[0][0]
                meta = getattr(gen, 'message', gen)
                usage = getattr(meta, 'response_metadata', {}).get('token_usage', None)
                if usage:
                    print("Token usage (from message):", usage)
                else:
                    print("Token usage not found. Full response:", response)
            except Exception:
                print("Token usage not found. Full response:", response)

handler = TokenUsageCallbackHandler()
llm = ChatOpenAI(model="gpt-3.5-turbo", callbacks=[handler])
llm.invoke("Tell me a joke.")

Token usage: {'completion_tokens': 16, 'prompt_tokens': 12, 'total_tokens': 28, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}


AIMessage(content="Why couldn't the bicycle stand up by itself?\n\nBecause it was two tired!", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 12, 'total_tokens': 28, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-Cgtw7CThCLCPCTc3tDryKizkbT8H7', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--18713641-5e9a-4341-a4a5-f171ffbea955-0', usage_metadata={'input_tokens': 12, 'output_tokens': 16, 'total_tokens': 28, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [19]:
# Collect All Prompts Callback: Store all prompts sent to the LLM
class PromptCollectorCallbackHandler(BaseCallbackHandler):
    
    def __init__(self):
        self.prompts = []
        
    def on_llm_start(self, prompts, **kwargs):
        self.prompts.extend(prompts)

collector = PromptCollectorCallbackHandler()

llm = ChatOpenAI(model="gpt-3.5-turbo", callbacks=[collector])
llm.invoke("What is LangChain?")
print("Prompts sent:", collector.prompts)

Error in PromptCollectorCallbackHandler.on_llm_start callback: TypeError('PromptCollectorCallbackHandler.on_llm_start() takes 2 positional arguments but 3 were given')


Prompts sent: []


In [20]:
# Error Logging Callback: Log errors during LLM execution
class ErrorLoggingCallbackHandler(BaseCallbackHandler):
    def on_llm_error(self, error, **kwargs):
        print(f"LLM error occurred: {error}")

handler = ErrorLoggingCallbackHandler()
llm = ChatOpenAI(model="gpt-3.5-turbo", callbacks=[handler])
try:
    llm.invoke(None)  # Intentionally cause an error
except Exception:
    pass

In [21]:
# Custom Progress Bar Callback: Show progress for multi-step chains
from tqdm import tqdm
class ProgressBarCallbackHandler(BaseCallbackHandler):
    def __init__(self, total):
        self.pbar = tqdm(total=total)
    def on_chain_start(self, *args, **kwargs):
        self.pbar.update(1)
    def on_chain_end(self, *args, **kwargs):
        self.pbar.close()

# Example usage for a chain with 3 steps
handler = ProgressBarCallbackHandler(total=3)
# Simulate chain steps
handler.on_chain_start(); handler.on_chain_start(); handler.on_chain_start(); handler.on_chain_end()

100%|██████████| 3/3 [00:00<00:00, 1493.88it/s]
